In [ ]:
%pip install -Uqqq transformers accelerate hf_transfer peft

In [ ]:
# %pip install torch torchvision

In [5]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = 'LGAI-EXAONE/EXAONE-4.0-1.2B'
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype = torch.bfloat16,
    device_map = 'auto'
)
tokenizer = AutoTokenizer.from_pretrained(model_id)


model.safetensors: reconstructing file:   0%|          |  0.00B / 2.56GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/332 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

Some parameters are on the meta device because they were offloaded to the disk and cpu.


...

vocab.json:   0%|          | 0.00/1.93M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.22M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/6.70k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.91M [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/5.49k [00:00<?, ?B/s]

In [6]:
# 샘플 데이터
prompt = '파이썬이 뭐야?'
chosen = '파이썬은 배우기 쉽고, 강력한 프로그래밍 언어입니다.'
rejected = '파이썬은 뱀의 일종이다.'

In [ ]:
# 생성확률 계싼

# 모델이 "이 답변이 얼마나 적절한지" 점수(log-prob 합)로 계산하는 함수
def get_logprob(model, tokenizer, prompt, response):
    full_text = prompt + response
    inputs = tokenizer(full_text, return_tensors='pt').to(model.device) # 토큰화 후 장치 이동
    prompt_len = len(tokenizer(prompt, return_tensors='pt')['input_ids'][0]) # 프롬프트 토큰 개수 확인

    with torch.no_grad():
        outputs = model(**inputs) # 순전파
        logits = outputs.logits   # 각 위치별 다음 토큰 후보들 점수들
        log_probs = F.log_softmax(logits, dim=-1) # 점수표 -> log확률표

        # 정답 토큰(실제로 다음에 나올 토큰)의 log확률을 위치별로 뽑아옴
        # seq_len의 마지막 위치는 그 다음 토큰을 맞출 정답인데 없어 길이맞춤용
        token_log_probs = log_probs[:, :-1].gather( # (batch, seq_len - 1, vocab_size)
            index = inputs['input_ids'][:, 1:].unsqueeze(-1), # (batch, seq_len-1, 1)
            dim = -1
        ).squeeze(-1)

        response_log_probs = token_log_probs[:, prompt_len - 1:]
        total = response_log_probs.sum()

    return total.item()
chosen_prob = get_logprob(model, tokenizer, prompt, chosen)
rejected_prob = get_logprob(model, tokenizer, prompt, rejected)
print(f"선호 답변 생성확률 : {chosen_prob}, 비선호 답변 생성 확률 : {rejected_prob}")

{'input_ids': tensor([[17830, 35344,   634,  2680,  1137,   392, 17830, 35344,   732,  5304,
           722,  3497,   853,   373,  9448,  1075, 42980, 10978, 10996,   375]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}
CausalLMOutputWithPast(loss=None, logits=tensor([[[-0.8398, -2.2031, -1.0391,  ..., -0.2949, -3.9375, -2.1875],
         [-1.0078, -0.5859, -1.2188,  ..., -1.6406, -3.1250, -2.0312],
         [-5.5000, -5.1250, -6.6875,  ..., -3.5469, -4.3750, -5.0938],
         ...,
         [-4.7188, -5.2188, -3.1719,  ..., -3.1719, -2.0312, -6.5938],
         [-2.3594, -3.2812, -2.1406,  ..., -2.1875, -3.1719, -3.3906],
         [-4.1250, -6.4375, -3.5781,  ..., -2.6562, -5.3438, -5.0312]]],
       dtype=torch.bfloat16), past_key_values=DynamicCache(layers=[DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, Dyn

In [12]:
# dpo 손실함수 : 정책모델, 참조모델의 차이를 작게 만드는 목적의 손실함수
# (정책모델이 선호답변에 준 log-prob점수, ...비선호 답변..., 참조 모델이 선호답변에 준 log_prob점수, ...비선호 답변..., beta : 선호를 강화시키는 개수)
def dpo_loss(policy_chosen, policy_rejected, ref_chosen, ref_rejected, beta=0.1):
    logits = beta * ((policy_chosen - policy_rejected) - (ref_chosen - ref_rejected))
    loss = -F.logsigmoid(torch.tensor(logits))
    return loss.item()